# Application Data Validation

This notebook validates synthetic recruiting applications.

The application table connects:

- Candidates
- Job requisitions
- Application outcomes
- Employees created from hired applications

The main rules are:

- Application IDs are complete, unique, and sequential.
- Candidate and requisition foreign keys are valid.
- Every candidate has at least one application.
- Candidate-requisition combinations are unique.
- Applications occur during the requisition period.
- Final applications have decision dates.
- In-process applications have no decision date.
- Offers are recorded only for hired and offer-declined applications.
- Hired applications map to all 10,000 employees.
- Filled requisition hiring counts agree with target headcount.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"


employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

candidates = pd.read_csv(
    RAW_DATA_DIR / "candidates.csv"
)

job_requisitions = pd.read_csv(
    RAW_DATA_DIR / "job_requisitions.csv",
    parse_dates=[
        "open_date",
        "close_date",
    ],
)

applications = pd.read_csv(
    RAW_DATA_DIR / "applications.csv",
    parse_dates=[
        "application_date",
        "offer_date",
        "decision_date",
    ],
)


print(
    "Employees:",
    employees.shape,
)

print(
    "Candidates:",
    candidates.shape,
)

print(
    "Job requisitions:",
    job_requisitions.shape,
)

print(
    "Applications:",
    applications.shape,
)

Employees: (10000, 15)
Candidates: (40000, 5)
Job requisitions: (2779, 9)
Applications: (52436, 9)


## 1. Initial inspection

In [2]:
applications.head(10)

,application_id,candidate_id,requisition_id,application_date,application_status,interview_score,offer_date,decision_date,employee_id
0,400001,200535,300003,2021-01-01,Hired,82.3,2021-01-03,2021-01-03,100535.0
1,400002,201321,300011,2021-01-01,Hired,78.5,2021-01-02,2021-01-17,101321.0
2,400003,201403,300006,2021-01-01,Hired,88.2,2021-01-01,2021-01-05,101403.0
3,400004,201505,300005,2021-01-01,Hired,88.4,2021-01-10,2021-02-02,101505.0
4,400005,201683,300006,2021-01-01,Hired,85.2,2021-01-01,2021-01-03,101683.0
5,400006,202046,300036,2021-01-01,Hired,91.9,2021-02-09,2021-02-11,102046.0
6,400007,202095,300034,2021-01-01,Hired,85.3,2021-01-01,2021-01-01,102095.0
7,400008,202223,300024,2021-01-01,Hired,79.3,2021-01-04,2021-01-07,102223.0
8,400009,202881,300032,2021-01-01,Hired,87.8,2021-01-03,2021-01-11,102881.0
9,400010,202920,300025,2021-01-01,Hired,75.0,2021-01-01,2021-01-01,102920.0


In [3]:
applications.info()

<class 'pandas.DataFrame'>
RangeIndex: 52436 entries, 0 to 52435
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   application_id      52436 non-null  int64         
 1   candidate_id        52436 non-null  int64         
 2   requisition_id      52436 non-null  int64         
 3   application_date    52436 non-null  datetime64[us]
 4   application_status  52436 non-null  str           
 5   interview_score     31001 non-null  float64       
 6   offer_date          13487 non-null  datetime64[us]
 7   decision_date       45026 non-null  datetime64[us]
 8   employee_id         10000 non-null  float64       
dtypes: datetime64[us](3), float64(2), int64(3), str(1)
memory usage: 3.6 MB


## 2. Structure and primary-key checks

In [4]:
expected_columns = [
    "application_id",
    "candidate_id",
    "requisition_id",
    "application_date",
    "application_status",
    "interview_score",
    "offer_date",
    "decision_date",
    "employee_id",
]

expected_application_ids = list(
    range(
        400_001,
        400_001
        + len(applications),
    )
)

structure_checks = pd.Series(
    {
        "table has nine columns": (
            applications.columns.tolist()
            == expected_columns
        ),
        "application IDs are complete": (
            applications[
                "application_id"
            ].notna().all()
        ),
        "application IDs are unique": (
            applications[
                "application_id"
            ].is_unique
        ),
        "application IDs are sequential": (
            applications[
                "application_id"
            ].astype(int).tolist()
            == expected_application_ids
        ),
        "candidate-requisition pairs are unique": (
            not applications
            .duplicated(
                subset=[
                    "candidate_id",
                    "requisition_id",
                ]
            )
            .any()
        ),
    },
    name="passed",
)

structure_checks

table has nine columns                    True
application IDs are complete              True
application IDs are unique                True
application IDs are sequential            True
candidate-requisition pairs are unique    True
Name: passed, dtype: bool

## 3. Foreign-key and coverage checks

In [5]:
foreign_key_checks = pd.Series(
    {
        "candidate IDs are valid": (
            set(
                applications[
                    "candidate_id"
                ]
            )
            .issubset(
                set(
                    candidates[
                        "candidate_id"
                    ]
                )
            )
        ),
        "requisition IDs are valid": (
            set(
                applications[
                    "requisition_id"
                ]
            )
            .issubset(
                set(
                    job_requisitions[
                        "requisition_id"
                    ]
                )
            )
        ),
        "employee IDs are valid": (
            set(
                applications[
                    "employee_id"
                ]
                .dropna()
                .astype(int)
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "every candidate has an application": (
            applications[
                "candidate_id"
            ].nunique()
            == len(candidates)
        ),
    },
    name="passed",
)

foreign_key_checks

candidate IDs are valid               True
requisition IDs are valid             True
employee IDs are valid                True
every candidate has an application    True
Name: passed, dtype: bool

## 4. Connect applications to requisitions

In [6]:
requisition_details = (
    job_requisitions[
        [
            "requisition_id",
            "job_role_id",
            "department_id",
            "location_id",
            "open_date",
            "close_date",
            "target_headcount",
            "requisition_status",
        ]
    ]
    .rename(
        columns={
            "open_date": (
                "requisition_open_date"
            ),
            "close_date": (
                "requisition_close_date"
            ),
        }
    )
)

application_details = (
    applications
    .merge(
        requisition_details,
        on="requisition_id",
        how="left",
    )
)

application_details[
    "requisition_end_date"
] = (
    application_details[
        "requisition_close_date"
    ]
    .fillna(
        pd.Timestamp("2026-06-30")
    )
)

application_details.head()

,application_id,candidate_id,requisition_id,application_date,application_status,interview_score,offer_date,decision_date,employee_id,job_role_id,department_id,location_id,requisition_open_date,requisition_close_date,target_headcount,requisition_status,requisition_end_date
0,400001,200535,300003,2021-01-01,Hired,82.3,2021-01-03,2021-01-03,100535.0,1,1,4,2021-01-01,2021-03-10,3,Filled,2021-03-10
1,400002,201321,300011,2021-01-01,Hired,78.5,2021-01-02,2021-01-17,101321.0,4,1,1,2021-01-01,2021-03-31,10,Filled,2021-03-31
2,400003,201403,300006,2021-01-01,Hired,88.2,2021-01-01,2021-01-05,101403.0,2,1,3,2021-01-01,2021-03-30,7,Filled,2021-03-30
3,400004,201505,300005,2021-01-01,Hired,88.4,2021-01-10,2021-02-02,101505.0,2,1,1,2021-01-01,2021-03-28,5,Filled,2021-03-28
4,400005,201683,300006,2021-01-01,Hired,85.2,2021-01-01,2021-01-03,101683.0,2,1,3,2021-01-01,2021-03-30,7,Filled,2021-03-30


## 5. Application-date checks

In [7]:
decided_applications = (
    application_details[
        application_details[
            "decision_date"
        ].notna()
    ]
)

offered_applications = (
    application_details[
        application_details[
            "offer_date"
        ].notna()
    ]
)

date_checks = pd.Series(
    {
        "applications occur after requisitions open": (
            application_details[
                "application_date"
            ]
            .ge(
                application_details[
                    "requisition_open_date"
                ]
            )
            .all()
        ),
        "applications occur before requisitions end": (
            application_details[
                "application_date"
            ]
            .le(
                application_details[
                    "requisition_end_date"
                ]
            )
            .all()
        ),
        "decisions occur after applications": (
            decided_applications[
                "decision_date"
            ]
            .ge(
                decided_applications[
                    "application_date"
                ]
            )
            .all()
        ),
        "decisions occur before requisitions end": (
            decided_applications[
                "decision_date"
            ]
            .le(
                decided_applications[
                    "requisition_end_date"
                ]
            )
            .all()
        ),
        "offers occur after applications": (
            offered_applications[
                "offer_date"
            ]
            .ge(
                offered_applications[
                    "application_date"
                ]
            )
            .all()
        ),
        "offers occur before decisions": (
            offered_applications[
                "offer_date"
            ]
            .le(
                offered_applications[
                    "decision_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

date_checks

applications occur after requisitions open    True
applications occur before requisitions end    True
decisions occur after applications            True
decisions occur before requisitions end       True
offers occur after applications               True
offers occur before decisions                 True
Name: passed, dtype: bool

## 6. Application-status checks

In [8]:
allowed_statuses = {
    "Hired",
    "Rejected",
    "Withdrawn",
    "Offer Declined",
    "In Process",
    "Position Cancelled",
}

hired_mask = (
    application_details[
        "application_status"
    ]
    == "Hired"
)

offer_declined_mask = (
    application_details[
        "application_status"
    ]
    == "Offer Declined"
)

in_process_mask = (
    application_details[
        "application_status"
    ]
    == "In Process"
)

position_cancelled_mask = (
    application_details[
        "application_status"
    ]
    == "Position Cancelled"
)

offer_mask = (
    hired_mask
    | offer_declined_mask
)

status_checks = pd.Series(
    {
        "statuses are valid": (
            set(
                applications[
                    "application_status"
                ]
            )
            .issubset(
                allowed_statuses
            )
        ),
        "all status categories appear": (
            set(
                applications[
                    "application_status"
                ]
            )
            == allowed_statuses
        ),
        "in-process applications have no decision date": (
            application_details.loc[
                in_process_mask,
                "decision_date",
            ].isna().all()
        ),
        "final applications have decision dates": (
            application_details.loc[
                ~in_process_mask,
                "decision_date",
            ].notna().all()
        ),
        "offers appear for hired or declined applications": (
            application_details.loc[
                offer_mask,
                "offer_date",
            ].notna().all()
        ),
        "other applications have no offer date": (
            application_details.loc[
                ~offer_mask,
                "offer_date",
            ].isna().all()
        ),
        "in-process applications use open requisitions": (
            application_details.loc[
                in_process_mask,
                "requisition_status",
            ].eq("Open").all()
        ),
        "hired applications use filled requisitions": (
            application_details.loc[
                hired_mask,
                "requisition_status",
            ].eq("Filled").all()
        ),
        "cancelled positions use cancelled requisitions": (
            application_details.loc[
                position_cancelled_mask,
                "requisition_status",
            ].eq("Cancelled").all()
        ),
    },
    name="passed",
)

status_checks

statuses are valid                                  True
all status categories appear                        True
in-process applications have no decision date       True
final applications have decision dates              True
offers appear for hired or declined applications    True
other applications have no offer date               True
in-process applications use open requisitions       True
hired applications use filled requisitions          True
cancelled positions use cancelled requisitions      True
Name: passed, dtype: bool

## 7. Interview-score checks

In [9]:
scored_applications = (
    applications[
        applications[
            "interview_score"
        ].notna()
    ]
)

hired_applications = (
    applications[
        applications[
            "application_status"
        ]
        == "Hired"
    ]
)

offer_declined_applications = (
    applications[
        applications[
            "application_status"
        ]
        == "Offer Declined"
    ]
)

score_checks = pd.Series(
    {
        "scores remain between 0 and 100": (
            scored_applications[
                "interview_score"
            ]
            .between(
                0,
                100,
            )
            .all()
        ),
        "hired applications have scores": (
            hired_applications[
                "interview_score"
            ].notna().all()
        ),
        "hired scores are at least 75": (
            hired_applications[
                "interview_score"
            ].ge(75).all()
        ),
        "offer-declined applications have scores": (
            offer_declined_applications[
                "interview_score"
            ].notna().all()
        ),
        "offer-declined scores are at least 70": (
            offer_declined_applications[
                "interview_score"
            ].ge(70).all()
        ),
    },
    name="passed",
)

score_checks

scores remain between 0 and 100            True
hired applications have scores             True
hired scores are at least 75               True
offer-declined applications have scores    True
offer-declined scores are at least 70      True
Name: passed, dtype: bool

## 8. Hired-application checks

In [10]:
non_hired_applications = (
    applications[
        applications[
            "application_status"
        ]
        != "Hired"
    ]
)

employee_mapping_checks = pd.Series(
    {
        "there are 10,000 hired applications": (
            len(hired_applications)
            == 10_000
        ),
        "hired applications contain employee IDs": (
            hired_applications[
                "employee_id"
            ].notna().all()
        ),
        "non-hired applications have no employee IDs": (
            non_hired_applications[
                "employee_id"
            ].isna().all()
        ),
        "hired employee IDs are unique": (
            hired_applications[
                "employee_id"
            ].is_unique
        ),
        "all employees have hired applications": (
            set(
                hired_applications[
                    "employee_id"
                ].astype(int)
            )
            == set(
                employees[
                    "employee_id"
                ]
            )
        ),
    },
    name="passed",
)

employee_mapping_checks

there are 10,000 hired applications            True
hired applications contain employee IDs        True
non-hired applications have no employee IDs    True
hired employee IDs are unique                  True
all employees have hired applications          True
Name: passed, dtype: bool

In [11]:
candidate_employee_mapping = (
    hired_applications[
        [
            "candidate_id",
            "employee_id",
            "requisition_id",
            "decision_date",
        ]
    ]
    .sort_values(
        "candidate_id"
    )
    .head(10)
)

candidate_employee_mapping

,candidate_id,employee_id,requisition_id,decision_date
13879,200001,100001.0,300920,2022-10-17
9735,200002,100002.0,300681,2022-04-03
2569,200003,100003.0,300199,2021-06-28
8405,200004,100004.0,300583,2022-03-24
17197,200005,100005.0,301134,2023-05-10
148,200006,100006.0,300018,2021-01-14
15030,200007,100007.0,300920,2022-11-28
12930,200008,100008.0,300920,2022-11-21
15186,200009,100009.0,300885,2022-12-06
5027,200010,100010.0,300347,2021-09-13


## 9. Filled-requisition hiring checks

In [12]:
hired_counts = (
    hired_applications
    .groupby(
        "requisition_id"
    )
    .size()
    .rename(
        "actual_hires"
    )
    .reset_index()
)

filled_requisition_alignment = (
    job_requisitions[
        job_requisitions[
            "requisition_status"
        ]
        == "Filled"
    ][
        [
            "requisition_id",
            "target_headcount",
        ]
    ]
    .merge(
        hired_counts,
        on="requisition_id",
        how="left",
    )
)

filled_requisition_alignment[
    "actual_hires"
] = (
    filled_requisition_alignment[
        "actual_hires"
    ]
    .fillna(0)
    .astype(int)
)

filled_headcount_checks = pd.Series(
    {
        "hired counts match target headcount": (
            filled_requisition_alignment[
                "actual_hires"
            ]
            .eq(
                filled_requisition_alignment[
                    "target_headcount"
                ]
            )
            .all()
        ),
        "total actual hires equal 10,000": (
            filled_requisition_alignment[
                "actual_hires"
            ].sum()
            == 10_000
        ),
    },
    name="passed",
)

filled_headcount_checks

hired counts match target headcount    True
total actual hires equal 10,000        True
Name: passed, dtype: bool

## 10. Application summary

In [13]:
application_status_summary = (
    applications
    .groupby(
        "application_status"
    )
    .agg(
        application_count=(
            "application_id",
            "count",
        ),
        candidates=(
            "candidate_id",
            "nunique",
        ),
        average_interview_score=(
            "interview_score",
            "mean",
        ),
    )
    .round(2)
)

application_status_summary

,application_count,candidates,average_interview_score
application_status,,,
Hired,10000,10000,87.98
In Process,7410,6965,72.53
Offer Declined,3487,3395,84.94
Position Cancelled,7321,6917,69.19
Rejected,18416,15835,61.42
Withdrawn,5802,5538,70.46


In [14]:
applications_per_candidate = (
    applications
    .groupby(
        "candidate_id"
    )
    .size()
)

applications_per_candidate_summary = (
    applications_per_candidate
    .value_counts()
    .sort_index()
    .rename_axis(
        "applications_per_candidate"
    )
    .reset_index(
        name="candidate_count"
    )
)

applications_per_candidate_summary

,applications_per_candidate,candidate_count
0,1,29601
1,2,8362
2,3,2037


## 11. Complete validation summary

In [15]:
all_checks = pd.concat(
    [
        structure_checks,
        foreign_key_checks,
        date_checks,
        status_checks,
        score_checks,
        employee_mapping_checks,
        filled_headcount_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,table has nine columns,True
1,application IDs are complete,True
2,application IDs are unique,True
3,application IDs are sequential,True
4,candidate-requisition pairs are unique,True
5,candidate IDs are valid,True
6,requisition IDs are valid,True
7,employee IDs are valid,True
8,every candidate has an application,True
9,applications occur after requisitions open,True


In [16]:
if validation_results[
    "passed"
].all():
    print(
        "All application validation "
        "checks passed."
    )
else:
    print(
        "One or more application "
        "validation checks failed."
    )

All application validation checks passed.


## 12. Conclusions

The synthetic application table successfully connects candidates, job requisitions, and hired employees.

### Successful checks

- Application IDs are complete, unique, and sequential.
- Candidate and requisition foreign keys are valid.
- Every candidate has at least one application.
- Candidate-requisition combinations are unique.
- Applications occur within requisition periods.
- Final applications contain decision dates.
- In-process applications have blank decision dates.
- Offers appear only for hired and offer-declined applications.
- Interview scores remain between 0 and 100.
- There are exactly 10,000 hired applications.
- Every employee is connected to one hired application.
- Non-hired applications do not contain employee IDs.
- Filled requisition hiring counts match target headcounts.
- All application validation checks passed.

### Current simplifications

- Hired candidates have one application in the current model.
- General candidates can have one to three applications.
- A candidate cannot apply to the same requisition twice.
- Interview scores are optional when no interview occurred.
- Application stages such as phone screen and technical interview are not stored separately.
- Offer compensation is not stored in the application table.
- All hired application decisions occur on the employee hire date.